In [1]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [2]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

In [4]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    "../data/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    "../data/valid",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

Found 2662 images belonging to 2 classes.
Found 442 images belonging to 2 classes.


In [6]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=5
)

Epoch 1/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 74s 828ms/step - accuracy: 0.9155 - loss: 0.1978 - val_accuracy: 0.9706 - val_loss: 0.0824
Epoch 2/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 74s 881ms/step - accuracy: 0.9737 - loss: 0.0707 - val_accuracy: 0.9819 - val_loss: 0.0651
Epoch 3/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 78s 926ms/step - accuracy: 0.9857 - loss: 0.0381 - val_accuracy: 0.9683 - val_loss: 0.0606
Epoch 4/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 78s 940ms/step - accuracy: 0.9906 - loss: 0.0231 - val_accuracy: 0.9864 - val_loss: 0.0616
Epoch 5/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 78s 933ms/step - accuracy: 0.9947 - loss: 0.0169 - val_accuracy: 0.9864 - val_loss: 0.0546


In [7]:
model.save("../models/final_model.h5")

In [8]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [9]:
model = load_model("../models/final_model.h5")

In [10]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    "../data/test",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 215 images belonging to 2 classes.


In [11]:
predictions = model.predict(test_data)
pred_classes = (predictions > 0.5).astype(int)

7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step  


In [12]:
true_classes = test_data.classes

In [13]:
cm = confusion_matrix(true_classes, pred_classes)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[118   3]
 [  3  91]]


In [14]:
print("\nClassification Report:\n")
print(classification_report(true_classes, pred_classes))


Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       121
           1       0.97      0.97      0.97        94

    accuracy                           0.97       215
   macro avg       0.97      0.97      0.97       215
weighted avg       0.97      0.97      0.97       215

